In [ ]:
import json
import pandas as pd


# Set up the data

In [ ]:
# https://github.com/openfootball/worldcup.json/blob/master/2026/worldcup.json

with open("data/worldcup.json") as f:
    j = json.load(f)

wc_matches = pd.DataFrame(j["matches"])

# Limit to group stage matches
wc_matches = wc_matches[wc_matches["round"].str.contains("Matchday")]

In [ ]:
with open("data/rankings.json") as f:
    j = json.load(f)

# normalize Results to a flat table
rankings = pd.json_normalize(j["Results"])


# extract human-readable team name (prefer en-GB if present)
def extract_name(name_list, locale="en-GB"):
    if not name_list:
        return None
    for entry in name_list:
        if entry.get("Locale") == locale:
            return entry.get("Description")
    return name_list[0].get("Description")


rankings["Team"] = rankings["TeamName"].apply(extract_name)

# choose useful columns and ensure numeric types
cols = [
    "Rank",
    "Team",
    "IdCountry",
    "ConfederationName",
    "TotalPoints",
    "PrevRank",
    "RankingMovement",
    "RatedMatches",
]
rankings = rankings[cols]
rankings["Rank"] = rankings["Rank"].astype(int)
rankings["TotalPoints"] = rankings["TotalPoints"].astype(float)


In [ ]:
prev_match = (
    pd.read_csv("data/results.csv")
    .astype({"date": "datetime64[s]"})
    .dropna(subset=["home_score", "away_score"])
)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49395,2026-06-09,Estonia,Lithuania,1.0,0.0,Baltic Cup,Tallinn,Estonia,False
49396,2026-06-09,Russia,Trinidad and Tobago,3.0,0.0,Friendly,Kaliningrad,Russia,False
49397,2026-06-09,Togo,Benin,5.0,1.0,Friendly,Lomé,Togo,False
49398,2026-06-09,Liberia,Sierra Leone,3.0,1.0,Friendly,Monrovia,Liberia,False


# Apply ranking score

In [53]:
rank_map = rankings[["Team", "TotalPoints"]].set_index("Team").to_dict()["TotalPoints"]
# ad hoc
rank_map["South Korea"] = rank_map["Korea Republic"]
rank_map["Czech Republic"] = rank_map["Czechia"]
rank_map["Bosnia & Herzegovina"] = rank_map["Bosnia and Herzegovina"]
rank_map["Ivory Coast"] = rank_map["Côte d'Ivoire"]
rank_map["Turkey"] = rank_map["Türkiye"]
rank_map["Iran"] = rank_map["IR Iran"]
rank_map["Cape Verde"] = rank_map["Cabo Verde"]
rank_map["DR Congo"] = rank_map["Congo DR"]


wc_matches["team1_rank"] = wc_matches["team1"].map(rank_map)
wc_matches["team2_rank"] = wc_matches["team2"].map(rank_map)
for t in wc_matches["team1"].unique():
    if t not in rank_map:
        print(f"Warning: {t} not found in rankings")

In [54]:
wc_matches.to_clipboard(index=False)

In [ ]:
# Largest diff is a 4-0
print((wc_matches["team1_rank"] - wc_matches["team2_rank"]).abs().max())


np.float64(503.6027979999999)